# 2. ESM-2: generate variants from a seed sequence

- **Input:** The ubiquitin amino acid sequence.
- **Model information:** Sequence context and patterns learned from natural protein sequences. The model does not receive a structure.
- **Design objective:** Generate variants with a specified number of substitutions using the model's amino acid preferences.

Sample three variants, compare their substitutions, and predict the structure of a selected variant. Ubiquitin's experimental structure provides a reference for inspection.


## Setup
Select **Runtime → Change runtime type → T4 GPU**, then run the two cells below to load the tools and models.


In [ ]:
!git -C BME305-protein-design-tools pull -q 2>/dev/null || git clone -q https://github.com/RomeroLab/BME305-protein-design-tools.git
%pip -q install --upgrade ./BME305-protein-design-tools
from protein_design_tools import load_pdb, load_ESMFold, load_ESM2, load_MPNN, load_RFdiffusion, predict_structure, MPNNdesign, ESM2design, RFdiffusion, alignment, view_structure, inspect_prediction, save_results


In [ ]:
load_ESM2()
load_ESMFold();


## 1. Load the starting protein
Load the ubiquitin sequence used for generation and view its experimental structure as a reference. ESM-2 uses only the sequence in the next step.


In [ ]:
native_seq, reference_pdb = load_pdb('1UBQ', chain='A')
view_structure(reference_pdb)


## 2. Generate variants
At each selected position, mask the residue and sample an amino acid from ESM-2's predictions. Each new sample becomes context for the next step. Higher temperature flattens the sampling distribution.


In [ ]:
designs = ESM2design(native_seq, num_sequences=3, n_mutations=5, temperature=1, seed=7)


## 3. Compare sequences
Compare the number and positions of substitutions. Examine changes in amino acid size, charge, or polarity. Mutation labels use positions starting at 1.


In [ ]:
alignment({'ubiquitin':native_seq, **designs})


## 4. Predict and compare structures
Predict a selected variant with ESMFold and compare it with ubiquitin's experimental backbone. ESMFold uses the sequence alone, without the reference coordinates. RMSD measures backbone agreement; pLDDT describes confidence in the prediction. Inspect the overall fold and regions of disagreement alongside their confidence.


In [ ]:
prediction_pdb = predict_structure(designs['design_1'], num_recycles=1)
metrics = inspect_prediction(reference_pdb, prediction_pdb)


## Discuss
- Does the predicted variant retain the overall fold? Are any changes near the sampled substitutions?
- How does sequence-only sampling differ from the backbone-conditioned ProteinMPNN demo?

## Save results
Download the designed sequences and structures for further analysis.


In [ ]:
bundle = save_results(designs, reference_pdb, prediction_pdb)
from google.colab import files
files.download(str(bundle))


## Reuse the tools
The same tools accept your own sequences and structures. See the [function reference](https://github.com/RomeroLab/BME305-protein-design-tools#function-reference) for use in later assignments.

## References
[ESM-2 and ESMFold](https://github.com/facebookresearch/esm) · [1UBQ](https://www.rcsb.org/structure/1UBQ)
